# Desafio Voxar Labs para IC's

* O desafio consiste em identificar 3 classes de tipos de estrada: asphalt, belgian blocks e off-road.  
* Os obstáculos que podem ser encontrados no caminho são: iluminação, chuva, ruído visual, informações na tela, etc. Outro desafio encontrado é o desbalanceamento entre as classes, a classe asphalt tem muito mais imagens para treino do que as demais classes.  
* Outros problemas podem surgir como os dados em formato de imagem, já que são formas bem diferentes de análise comparado à planilhas.

## Abordagem para a solução

Para resolver o problema da classificação de superfícies em três classes, foi adotada uma abordagem com redes neurais convolucionais pré-treinadas.  
Foram testadas duas arquiteturas amplamente utilizadas em visão computacional:

* ResNet18
* EfficientNet-B0

Os modelos foram carregados com pesos pré-treinados no dataset ImageNet, e suas camadas finais de classificação foram substituídas por uma nova camada totalmente conectada com 3 saídas, correspondente às classes do desafio. Estes modelos foram utilizados por causa da redução do tempo de treinamento, boa performance em datasets menores, fácil implementação e aproveitamento de padrões já aprendidos em grandes bases de dados.

## Bibliotecas Utilizadas:

1. PyTorch: treinamento e manipulação dos modelos;
2. Torchvision: datasets, transforms e modelos pré-treinados;
3. Scikit-learn: métricas de avaliação;
4. Matplotlib: visualização de resultados;
5. NumPy: operações auxiliares.

## Preparação e execução

O dataset fornecido já estava organizado em diretórios separados para treino e teste, com subpastas representando cada classe. Para carregar as imagens, foi utilizada a classe ImageFolder do Torchvision, que associa automaticamente cada pasta a um rótulo numérico.

Também foram aplicadas transformações de pré-processamento nas imagens:

* redimensionamento para tamanho compatível com os modelos;
* conversão para tensor;
* normalização com médias e desvios padrão do ImageNet.

Em seguida, os dados foram organizados em batches por meio do DataLoader, permitindo treinamento eficiente em lotes.

## Treinamento

Os modelos foram treinados por 10 épocas, utilizando:

* otimizador Adam;
* função de perda CrossEntropyLoss;
* execução em GPU, quando disponível.

Durante o treinamento, a loss de cada época foi registrada para acompanhamento da convergência.

## Solução Baseline:

In [1]:
# Importação das bibliotecas e inicialização dos dados
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, classification_report
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device) # Onde o programa vai ser executado, se existir uma gpu compatível com cuda (nvidia) disponível, usa a GPU (mais rapido), se não vai para cpu

train_path = "dataset_processed/train"
test_path  = "dataset_processed/test"

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

cpu


In [2]:
# Dados de Treino
train_dataset = datasets.ImageFolder(train_path, transform=train_transform)
test_dataset  = datasets.ImageFolder(test_path, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

classes = train_dataset.classes
print(classes)

['asphalt', 'belgian_blocks', 'offroad']


In [3]:
# Função Treino
def train_model(model, epochs=10):

    model.to(device)

    criterion = nn.CrossEntropyLoss() # mede quão longe a previsão do modelo está da resposta correta (padrão para multiclasse)
    optimizer = optim.Adam(model.parameters(), lr=0.001) # atualiza os pesos da rede para reduzir o erro

    start = time.time()

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad() # zera gradientes antigos
            outputs = model(images) # as imagens entram no modelo
            loss = criterion(outputs, labels) # compara as saidas com o gabarito e calcula a loss
            loss.backward() # calcuça quanto cada peso contribuiu para o erro
            optimizer.step() # atualiza os pesos

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss:.4f}")

    end = time.time()

    print("Tempo treino:", round(end-start,2), "segundos")

In [4]:
# Função Teste
def evaluate_model(model):

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images) # gera as previsões
            _, preds = torch.max(outputs, 1) # escolhe a classe com o maior valor

            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred) # calcula a porcentaggem de acertos

    print("Accuracy:", accuracy)
    print(classification_report(y_true, y_pred, target_names=classes)) # mostra as métricas por classe

In [5]:
# Avaliação do ResNet18
print("\n##### RESNET18 #####")

resnet = models.resnet18(weights="DEFAULT") # carrega o modelo
resnet.fc = nn.Linear(resnet.fc.in_features, 3) # muda para 3 classes

train_model(resnet, epochs=10) # treina
evaluate_model(resnet) # testa

# Avaliação do EfficientNet-B0
print("\n##### EFFICIENTNET_B0 #####")

effnet = models.efficientnet_b0(weights="DEFAULT")
effnet.classifier[1] = nn.Linear(
    effnet.classifier[1].in_features, 3
)

train_model(effnet, epochs=10)
evaluate_model(effnet)


##### RESNET18 #####
Epoch 1/10 Loss: 10.3214
Epoch 2/10 Loss: 4.5870
Epoch 3/10 Loss: 4.9802
Epoch 4/10 Loss: 4.0146
Epoch 5/10 Loss: 3.9685
Epoch 6/10 Loss: 3.2617
Epoch 7/10 Loss: 2.6079
Epoch 8/10 Loss: 2.4701
Epoch 9/10 Loss: 6.0351
Epoch 10/10 Loss: 3.5111
Tempo treino: 602.89 segundos
Accuracy: 0.86
                precision    recall  f1-score   support

       asphalt       0.85      1.00      0.92       218
belgian_blocks       1.00      0.03      0.06        32
       offroad       0.89      0.80      0.84        50

      accuracy                           0.86       300
     macro avg       0.91      0.61      0.61       300
  weighted avg       0.88      0.86      0.81       300


##### EFFICIENTNET_B0 #####
Epoch 1/10 Loss: 6.9705
Epoch 2/10 Loss: 2.1521
Epoch 3/10 Loss: 1.2590
Epoch 4/10 Loss: 2.6530
Epoch 5/10 Loss: 4.0699
Epoch 6/10 Loss: 1.4209
Epoch 7/10 Loss: 0.5798
Epoch 8/10 Loss: 0.4180
Epoch 9/10 Loss: 0.1343
Epoch 10/10 Loss: 0.2099
Tempo treino: 488.44 segun

## Analise dos Resultados:

A baseline inicial utilizou:
```
learning rate = 0.001
batch size = 32
CrossEntropyLoss padrão
10 épocas
```

### ResNet18:

| Métrica     | Valor |
| ----------- | ----- |
| Accuracy    | 0.86  |
| Macro F1    | 0.61  |
| Weighted F1 | 0.81  |

* O modelo apresentou bom desempenho global em accuracy, porém com forte desequilíbrio entre classes.

| Classe         | Precision | Recall | F1-score |
| -------------- | --------- | ------ | -------- |
| Asphalt        | 0.85      | 1.00   | 0.92     |
| Belgian Blocks | 1.00      | 0.03   | 0.06     |
| Off-road       | 0.89      | 0.80   | 0.84     |

1. Asphalt foi quase perfeita, todas as imagens reais de Asphalt foram detectadas.
2. Pela recall de belgian blocks praticamente todas as imagens dessa classe foram classificadas incorretamente.

* O modelo ficou enviesado para a classe majoritária, falhando em reconhecer Belgian Blocks.


### EfficientNet-B0

| Métrica     | Valor  |
| ----------- | ------ |
| Accuracy    | 0.9033 |
| Macro F1    | 0.77   |
| Weighted F1 | 0.89   |

* O EfficientNet-B0 apresentou baseline claramente superior ao ResNet18.

| Classe         | Precision | Recall | F1-score |
| -------------- | --------- | ------ | -------- |
| Asphalt        | 0.95      | 0.98   | 0.96     |
| Belgian Blocks | 0.86      | 0.38   | 0.52     |
| Off-road       | 0.75      | 0.92   | 0.83     |

1. Excelente desempenho da asphalt.
2. Belgian blocks ainda difícil, mas bem melhor que no modelo anterior.
3. Off-road teve ótimo desempenho.

#### Comparação entre os modelos: 

| Métrica        | ResNet18 | EfficientNet-B0 |
| -------------- | -------- | --------------- |
| Accuracy       | 0.86     | **0.9033**      |
| Macro F1       | 0.61     | **0.77**        |
| Belgian Recall | 0.03     | **0.38**        |


EfficientNet-B0 foi superior na baseline.

## Experimentos:

### Experimento 1 – Uso do Parâmetro Weights

Mudança:
```
CrossEntropyLoss(weight=...)
```
Objetivo:
Reduzir impacto do desbalanceamento e penalizar mais erros em classes minoritárias

In [6]:
# Função Treino
def train_model(model, epochs=10):

    model.to(device)

    pesos = torch.tensor([1.0, 4.0, 2.5], dtype=torch.float32).to(device) # MUDANÇA

    criterion = nn.CrossEntropyLoss(weight=pesos) # mede quão longe a previsão do modelo está da resposta correta (padrão para multiclasse)
    optimizer = optim.Adam(model.parameters(), lr=0.001) # atualiza os pesos da rede para reduzir o erro

    start = time.time()

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad() # zera gradientes antigos
            outputs = model(images) # as imagens entram no modelo
            loss = criterion(outputs, labels) # compara as saidas com o gabarito e calcula a loss
            loss.backward() # calcuça quanto cada peso contribuiu para o erro
            optimizer.step() # atualiza os pesos

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss:.4f}")

    end = time.time()

    print("Tempo treino:", round(end-start,2), "segundos")

In [7]:
# Avaliação do ResNet18
print("\n##### RESNET18 #####")

resnet = models.resnet18(weights="DEFAULT") # carrega o modelo
resnet.fc = nn.Linear(resnet.fc.in_features, 3) # muda para 3 classes

train_model(resnet, epochs=10) # treina
evaluate_model(resnet) # testa

# Avaliação do EfficientNet-B0
print("\n##### EFFICIENTNET_B0 #####")

effnet = models.efficientnet_b0(weights="DEFAULT")
effnet.classifier[1] = nn.Linear(
    effnet.classifier[1].in_features, 3
)

train_model(effnet, epochs=10)
evaluate_model(effnet)


##### RESNET18 #####
Epoch 1/10 Loss: 13.2073
Epoch 2/10 Loss: 7.5981
Epoch 3/10 Loss: 5.8278
Epoch 4/10 Loss: 4.6344
Epoch 5/10 Loss: 3.4727
Epoch 6/10 Loss: 3.7219
Epoch 7/10 Loss: 2.3704
Epoch 8/10 Loss: 4.5980
Epoch 9/10 Loss: 5.3959
Epoch 10/10 Loss: 4.2685
Tempo treino: 472.22 segundos
Accuracy: 0.8766666666666667
                precision    recall  f1-score   support

       asphalt       0.90      0.95      0.93       218
belgian_blocks       0.55      0.38      0.44        32
       offroad       0.91      0.86      0.89        50

      accuracy                           0.88       300
     macro avg       0.79      0.73      0.75       300
  weighted avg       0.86      0.88      0.87       300


##### EFFICIENTNET_B0 #####
Epoch 1/10 Loss: 8.4892
Epoch 2/10 Loss: 3.7116
Epoch 3/10 Loss: 2.0903
Epoch 4/10 Loss: 1.6159
Epoch 5/10 Loss: 1.2295
Epoch 6/10 Loss: 3.3583
Epoch 7/10 Loss: 2.8110
Epoch 8/10 Loss: 3.9085
Epoch 9/10 Loss: 2.7482
Epoch 10/10 Loss: 1.1351
Tempo treino

## Analise dos Resultados:

### ResNet18: Baseline x Experimento 1:

| Métrica     | Baseline | Exp.1  |
| ----------- | -------- | ------ |
| Accuracy    | 0.86     | 0.8767 |
| Macro F1    | 0.61     | 0.75   |
| Weighted F1 | 0.81     | 0.87   |

### Recall por classe:

| Classe         | Baseline | Exp.1 |
| -------------- | -------- | ----- |
| Asphalt        | 1.00     | 0.95  |
| Belgian Blocks | 0.03     | 0.38  |
| Off-road       | 0.80     | 0.86  |

* O experimento corrigiu a principal fraqueza do modelo.


### EfficientNet-B0: Baseline x Experimento 1:

| Métrica     | Baseline | Exp.1 |
| ----------- | -------- | ----- |
| Accuracy    | 0.9033   | 0.94  |
| Macro F1    | 0.77     | 0.88  |
| Weighted F1 | 0.89     | 0.94  |

### Recall por classe:

| Classe         | Baseline | Exp.1 |
| -------------- | -------- | ----- |
| Asphalt        | 0.98     | 0.98  |
| Belgian Blocks | 0.38     | 0.69  |
| Off-road       | 0.92     | 0.94  |

* EfficientNet-B0 com class weights melhorou ainda mais e foi o melhor modelo até agora.

### Experimento 2 – Redução da Learning Rate

Mudança:
```
0.001 → 0.0001
```
Objetivo:
Reduzir oscilações observadas na loss e tornar o treinamento mais estável.

In [8]:
# Função Treino
def train_model(model, epochs=10):

    model.to(device)

    criterion = nn.CrossEntropyLoss() # mede quão longe a previsão do modelo está da resposta correta (padrão para multiclasse)
    optimizer = optim.Adam(model.parameters(), lr=0.0001) # MUDANÇA

    start = time.time()

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad() # zera gradientes antigos
            outputs = model(images) # as imagens entram no modelo
            loss = criterion(outputs, labels) # compara as saidas com o gabarito e calcula a loss
            loss.backward() # calcuça quanto cada peso contribuiu para o erro
            optimizer.step() # atualiza os pesos

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss:.4f}")

    end = time.time()

    print("Tempo treino:", round(end-start,2), "segundos")

In [9]:
# Avaliação do ResNet18
print("\n##### RESNET18 #####")

resnet = models.resnet18(weights="DEFAULT") # carrega o modelo
resnet.fc = nn.Linear(resnet.fc.in_features, 3) # muda para 3 classes

train_model(resnet, epochs=10) # treina
evaluate_model(resnet) # testa

# Avaliação do EfficientNet-B0
print("\n##### EFFICIENTNET_B0 #####")

effnet = models.efficientnet_b0(weights="DEFAULT")
effnet.classifier[1] = nn.Linear(
    effnet.classifier[1].in_features, 3
)

train_model(effnet, epochs=10)
evaluate_model(effnet)


##### RESNET18 #####
Epoch 1/10 Loss: 8.6121
Epoch 2/10 Loss: 1.4084
Epoch 3/10 Loss: 0.4840
Epoch 4/10 Loss: 0.7350
Epoch 5/10 Loss: 1.6936
Epoch 6/10 Loss: 0.8265
Epoch 7/10 Loss: 0.5363
Epoch 8/10 Loss: 1.1065
Epoch 9/10 Loss: 0.6355
Epoch 10/10 Loss: 0.3314
Tempo treino: 469.81 segundos
Accuracy: 0.9066666666666666
                precision    recall  f1-score   support

       asphalt       0.92      0.98      0.95       218
belgian_blocks       0.87      0.41      0.55        32
       offroad       0.85      0.92      0.88        50

      accuracy                           0.91       300
     macro avg       0.88      0.77      0.80       300
  weighted avg       0.90      0.91      0.90       300


##### EFFICIENTNET_B0 #####
Epoch 1/10 Loss: 19.6539
Epoch 2/10 Loss: 6.6301
Epoch 3/10 Loss: 2.7342
Epoch 4/10 Loss: 1.5882
Epoch 5/10 Loss: 1.0272
Epoch 6/10 Loss: 0.7445
Epoch 7/10 Loss: 0.5096
Epoch 8/10 Loss: 0.3622
Epoch 9/10 Loss: 0.3245
Epoch 10/10 Loss: 0.1962
Tempo treino

## Analise dos Resultados:

### ResNet18: Baseline x Experimento 2:

| Métrica     | Baseline | Exp.2  |
| ----------- | -------- | ------ |
| Accuracy    | 0.86     | 0.9067 |
| Macro F1    | 0.61     | 0.80   |
| Weighted F1 | 0.81     | 0.90   |


### Recall por classe:

| Classe         | Baseline | Exp.2 |
| -------------- | --------------- | ------------ |
| Asphalt        | 1.00            | 0.98         |
| Belgian Blocks | 0.03            | 0.41         |
| Off-road       | 0.80            | 0.92         |

* A curva da loss ficou estável e houve um ganho forte nas classes minoritárias.


### EfficientNet-B0: Baseline x Experimento 2:

| Métrica     | Baseline | Exp.2 |
| ----------- | -------- | ----- |
| Accuracy    | 0.9033   | 0.93  |
| Macro F1    | 0.77     | 0.85  |
| Weighted F1 | 0.89     | 0.92  |

### Recall por classe:

| Classe         | Baseline | Exp.2 |
| -------------- | --------------- | ------------ |
| Asphalt        | 0.98            | 0.99         |
| Belgian Blocks | 0.38            | 0.56         |
| Off-road       | 0.92            | 0.90         |

* A learning rate menor refinou o treinamento e melhorou o desempenho.

### Experimento 3 – Aumento do Batch Size

Mudança:
```
32 → 64
```
Objetivo:
Verificar se batches maiores melhorariam estabilidade e eficiência

In [10]:
# Dados de Treino
train_dataset = datasets.ImageFolder(train_path, transform=train_transform)
test_dataset  = datasets.ImageFolder(test_path, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True) # MUDANÇA
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False) # MUDANÇA

classes = train_dataset.classes
print(classes)

['asphalt', 'belgian_blocks', 'offroad']


In [11]:
# Função Treino
def train_model(model, epochs=10):

    model.to(device)

    criterion = nn.CrossEntropyLoss() # mede quão longe a previsão do modelo está da resposta correta (padrão para multiclasse)
    optimizer = optim.Adam(model.parameters(), lr=0.001) # atualiza os pesos da rede para reduzir o erro

    start = time.time()

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad() # zera gradientes antigos
            outputs = model(images) # as imagens entram no modelo
            loss = criterion(outputs, labels) # compara as saidas com o gabarito e calcula a loss
            loss.backward() # calcuça quanto cada peso contribuiu para o erro
            optimizer.step() # atualiza os pesos

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss:.4f}")

    end = time.time()

    print("Tempo treino:", round(end-start,2), "segundos")

In [12]:
# Avaliação do ResNet18
print("\n##### RESNET18 #####")

resnet = models.resnet18(weights="DEFAULT") # carrega o modelo
resnet.fc = nn.Linear(resnet.fc.in_features, 3) # muda para 3 classes

train_model(resnet, epochs=10) # treina
evaluate_model(resnet) # testa

# Avaliação do EfficientNet-B0
print("\n##### EFFICIENTNET_B0 #####")

effnet = models.efficientnet_b0(weights="DEFAULT")
effnet.classifier[1] = nn.Linear(
    effnet.classifier[1].in_features, 3
)

train_model(effnet, epochs=10)
evaluate_model(effnet)


##### RESNET18 #####
Epoch 1/10 Loss: 5.3232
Epoch 2/10 Loss: 2.9667
Epoch 3/10 Loss: 4.4632
Epoch 4/10 Loss: 2.1737
Epoch 5/10 Loss: 1.5981
Epoch 6/10 Loss: 2.5052
Epoch 7/10 Loss: 5.4032
Epoch 8/10 Loss: 2.0214
Epoch 9/10 Loss: 4.8636
Epoch 10/10 Loss: 2.2620
Tempo treino: 443.42 segundos
Accuracy: 0.8433333333333334
                precision    recall  f1-score   support

       asphalt       0.84      0.99      0.91       218
belgian_blocks       0.64      0.22      0.33        32
       offroad       0.97      0.62      0.76        50

      accuracy                           0.84       300
     macro avg       0.81      0.61      0.66       300
  weighted avg       0.84      0.84      0.82       300


##### EFFICIENTNET_B0 #####
Epoch 1/10 Loss: 3.8531
Epoch 2/10 Loss: 1.1025
Epoch 3/10 Loss: 1.1976
Epoch 4/10 Loss: 0.9143
Epoch 5/10 Loss: 0.6329
Epoch 6/10 Loss: 0.5433
Epoch 7/10 Loss: 0.2989
Epoch 8/10 Loss: 0.1800
Epoch 9/10 Loss: 0.0248
Epoch 10/10 Loss: 0.4955
Tempo treino:

## Analise dos Resultados:

### ResNet18: Baseline vs Experimento 3:

| Métrica     | Baseline | Exp.3  |
| ----------- | -------- | ------ |
| Accuracy    | 0.86     | 0.8433 |
| Macro F1    | 0.61     | 0.66   |
| Weighted F1 | 0.81     | 0.82   |

### Recall por classe:

| Classe         | Recall Baseline | Exp.3 |
| -------------- | --------------- | ----- |
| Asphalt        | 1.00            | 0.99  |
| Belgian Blocks | 0.03            | 0.22  |
| Off-road       | 0.80            | 0.62  |

* O batch size maior não foi benéfico para o modelo. Houve pequena melhora em belgian blocks, porém com perda geral de desempenho. A loss continuou bastante oscilante, sugerindo que o batch maior não resolveu a instabilidade do treinamento.

### EfficientNet-B0: Baseline vs Experimento 3:

| Métrica     | Baseline | Exp.3 |
| ----------- | -------- | ----- |
| Accuracy    | 0.9033   | 0.92  |
| Macro F1    | 0.77     | 0.80  |
| Weighted F1 | 0.89     | 0.91  |

### Recall por classe:

| Classe         | Recall Baseline | Exp.3 |
| -------------- | --------------- | ----- |
| Asphalt        | 0.98            | 0.99  |
| Belgian Blocks | 0.38            | 0.41  |
| Off-road       | 0.92            | 0.96  |

* O batch size maior foi positivo para EfficientNet-B0, especialmente no desempenho geral e na classe off-road.

## Conclusões dos experimentos:

1. EfficientNet-B0 foi superior ao ResNet18 na maioria dos cenários.
2. Dataset desbalanceado impacta fortemente o modelo, sem correção o modelo favorece a classe asphalt e ignora a classe belgian blocks.
3. A class weights funcionou bem, melhorou o equilíbrio e recall nas classes minoritárias.
4. Reduzir a learning rate melhorou estabilidade e desempenho.
5. O aumento do batch size teve efeito dependente da arquitetura, no ResNet18 ele perdeu desempenho e no EfficientNet-B0 ele melhorou moderadamente.

## Melhor Solução Final:

### EfficientNet-B0 + CrossEntropyLoss com pesos por classe

```
Modelo: EfficientNet-B0
Loss: CrossEntropyLoss(weight=class_weights)
Learning Rate: 0.001
Batch Size: 32
Épocas: 10
```

Essa configuração foi considerada a melhor porque apresentou:

* Maior accuracy entre todos os testes;
* Melhor equilíbrio entre classes;
* Melhor recall para belgian blocks;
* Excelente desempenho em asphalt e off-road;
* Boa robustez geral.

Além disso, mostrou que o principal ponto do problema era o desbalanceamento dos dados, e não necessariamente a arquitetura.

## Próximos passos:

* Adaptar para previsão em tempo real
* Aplicar a Gaussian Blur
* Observar Matriz de confusão
* Separar imagens para teste e observar os desafios que o modelo enfrenta para a identificação

## LLMS usadas

Utilizei o ChatGPT 5.4 mini para apoio conceitual, debugging e organização experimental.